In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:
from grupo10.utils.logger import get_logger
from grupo10.utils.audit import ensure_audit_table, write_audit_record
#Databricks ja importa spark globalmente, mas pylance no VSC grita se não importar deliberadamente.
from databricks.sdk.runtime import spark

#TODO definir a a string do nome da tabela de audit em outro local e referenciar aqui?
#O underline antes do nome da tabela de auditoria de ingestão é uma convenção para dizer que ela é interna, ou seja, não é uma tabela do negócio em si
AUDIT_TABLE = "lh_nautical.bronze._ingestion_audit"

logger = get_logger("sqlserver_ingest")
ensure_audit_table(spark, AUDIT_TABLE)
logger.info(f"Audit table ready: {AUDIT_TABLE}")


In [ ]:
import time

from databricks.sdk.runtime import dbutils, spark

# Dados da conexão - Ideal é estarem numa variável de ambiente e serem puxadas de lá
db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"

# Lista das tabelas que vão ser extraídas - Geralmente é obtido através de outro banco de dados que chama de ControlDB, mas aqui no nosso projeto não vejo problema de apenas colocarmos eles num JSON dentro do databricks
target_tables = [
    "customers",
    "ordersss",
#   "order_items",
#   "products",
#   "product_variants",
#   "brands",
#   "categories",
#   "returns",
#   "return_items",
#   "stock_levels",
#   "stock_movements",
#   "suppliers",
#   "employees",
  "locatiozzns",
    "addresses",
]

# Loop para processar cada tabela baseando-se nos dados da conexão declarados anteriormente
failed_tables = []

for table in target_tables:
    start = time.time()
    try:
        logger.info(f"Starting extraction: {table}")

        jdbc_options = {
            "url": jdbc_url,
            "dbtable": table,
            "user": db_user,
            "password": db_password,
            "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
            "encrypt": "true",
            "trustServerCertificate": "true",
            "loginTimeout": "30",
            "fetchsize": "1000",
        }

        # Lendo a tabela atual do SQL Server
        df_temp = spark.read.format("jdbc").options(**jdbc_options).load()

        output_table = f"lh_nautical.bronze.{table}"
        #TODO verificar se este overwrite não deveria ser append
        df_temp.write.format("delta").mode("overwrite").saveAsTable(output_table)

        row_count = df_temp.count()
        duration = time.time() - start
        logger.info(f"Done: {table} | rows={row_count} | duration={duration:.2f}s")

        write_audit_record(
            spark=spark,
            table_name=table,
            status="success",
            duration_seconds=duration,
            row_count=row_count,
            audit_table=AUDIT_TABLE,
        )

    except Exception as e:
        duration = time.time() - start
        logger.error(f"Failed: {table} | duration={duration:.2f}s | error={e}")
        failed_tables.append(table)

        write_audit_record(
            spark=spark,
            table_name=table,
            status="failed",
            duration_seconds=duration,
            error_message=str(e),
            audit_table=AUDIT_TABLE,
        )

if failed_tables:
    logger.warning(f"Ingestion finished with {len(failed_tables)} failure(s): {failed_tables}")
else:
    logger.info("All tables ingested successfully.")
